In [2]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' can be imported
import sys
from pathlib import Path # for path manipulations
# Move three levels up from current working directory to reach workspace root
workspace_root = Path.cwd().parent.parent.parent.resolve() 
smx_dir = workspace_root / 'smx'  # Path to smx folder
if str(smx_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(smx_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{workspace_root}/real_datasets/xrf/bank_notes.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '2.74':'22.71']

# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '2.74':'22.71'], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '2.74':'22.71'], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# preprocessings
import preprocessings as prepr  # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-26 11:19:40,717 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-26 11:19:40,785 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

In [3]:
# PLS-DA with optimized latent variables
from modeling import pls_optimized

plsda_results = pls_optimized(
    Xcalclass_prep, 
    ycalclass,
    LVmax=4,
    Xpred=Xpredclass_prep,
    ypred=ypredclass,
    aim='classification',
    cv=10
)

# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

# plotando o vip scores rapidamente
vip_scores_mat.T.plot()

## Definição das Zonas Espectrais

In [4]:
spectral_cuts = [
('Ar ka + Ag L', 2.76, 3.47),
('Ca ka', 3.5, 3.91),
('Ca kb', 3.93, 4.24),
('Ti ka', 4.26, 4.72),
('Ti kb', 4.75, 5.13),
('background1', 5.16, 6.12),
('Fe ka', 6.15, 6.76),
('Fe kb', 6.79, 7.32),
('background2', 7.35, 7.78),
('Cu ka', 7.81, 8.29),
('Zn ka', 8.29, 8.80),
('Cu kb', 8.80, 9.26),
('Zn kb', 9.26, 10.00),
('background3', 10.00, 21.46),
('Ag ka scattering', 21.49, 22.71)
]

## VIP, Regression Coefficients e SHAP (como no original)

In [5]:
# VIP scores por energia
vip_scores_df = pd.DataFrame({
    'energy': vip_scores_mat.T.index,
    'VIP_Score': vip_scores_mat.T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in vip_scores_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip)
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# Coeficientes de regressão do PLS
reg_vet = pd.DataFrame(pls_model.coef_, columns=pls_model.feature_names_in_).T
reg_vet.insert(0, 'energy', reg_vet.index)
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy','Reg_coef']
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs()
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)
energy_to_zone_reg = {}
for zone_name, start, end in spectral_cuts:
    for e in reg_vet['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_reg[e] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg)
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

In [6]:
shap_global_importance = pd.read_csv('shap_bank_notes.csv', sep=';') # loading previously saved shap_unique_df

# vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_shap = {}
for zone_name, start, end in spectral_cuts:
    for i in shap_global_importance['energy']:
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_shap[i] = zone_name
shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
shap_unique_df = shap_global_importance.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
shap_unique_df = shap_unique_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
shap_unique_df

,energy,Mean_Abs_SHAP,Zone
0,4.52,0.053898,Ti ka
1,3.73,0.019426,Ca ka
2,8.06,0.004351,Cu ka
3,6.46,0.002868,Fe ka
4,4.95,0.001924,Ti kb
5,21.92,0.000429,Ag ka scattering
6,4.03,0.000019,Ca kb
7,17.23,0.000007,background3
8,2.76,0.000000,Ar ka + Ag L
9,6.79,0.000000,Fe kb


## Funções de Agregação por PCA

Aqui implementamos as funções para agregar zonas espectrais usando PCA com 1 componente principal.

In [7]:
import explaining as exp

# Extração das zonas espectrais
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_scores_df, pca_info_dict = exp.aggregate_spectral_zones_pca(spectral_zones_class) # apca aggregation
print(f"\nScores DataFrame shape: {zone_scores_df.shape}")

Zona 'Ar ka + Ag L': VE = 5.85%, dim = 29 variáveis
Zona 'Ca ka': VE = 98.41%, dim = 17 variáveis
Zona 'Ca kb': VE = 90.24%, dim = 13 variáveis
Zona 'Ti ka': VE = 96.36%, dim = 19 variáveis
Zona 'Ti kb': VE = 80.62%, dim = 16 variáveis
Zona 'background1': VE = 12.07%, dim = 39 variáveis
Zona 'Fe ka': VE = 99.63%, dim = 25 variáveis
Zona 'Fe kb': VE = 96.66%, dim = 22 variáveis
Zona 'background2': VE = 8.40%, dim = 18 variáveis
Zona 'Cu ka': VE = 97.03%, dim = 20 variáveis
Zona 'Zn ka': VE = 20.69%, dim = 21 variáveis
Zona 'Cu kb': VE = 63.49%, dim = 19 variáveis
Zona 'Zn kb': VE = 5.72%, dim = 30 variáveis
Zona 'background3': VE = 2.51%, dim = 451 variáveis
Zona 'Ag ka scattering': VE = 27.73%, dim = 49 variáveis

Scores DataFrame shape: (284, 15)


In [8]:
predicates_quantiles = exp.predicates_by_quantiles(zone_scores_df, [0.2, 0.4, 0.6, 0.8])
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_scores_df,
    y_predicted_numeric=y_pred_cont
)

In [9]:
# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_pred_conticted_numeric = pd.Series(y_pred_cont) # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred_conticted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.01, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graph(
        bags_result=all_results_cov[seed]['bags_result'],
        predicate_ranking_dict=all_results_cov[seed]['cov_results_dict'],
        metric_column='Covariance',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.01

Processando semente: 1

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados

,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2,Predicate_Cov_Seed_3
0,Fe ka > -23.71,Fe ka > -23.71,Fe ka > -23.71,Fe ka > -23.71
1,Fe ka > -24.15,Fe ka > -24.15,Fe ka > -24.15,Fe ka > -24.15
2,Fe ka > -24.62,Fe ka > -24.62,Fe ka > -24.62,Fe ka > -24.62
3,Ca ka > -26.11,Ca ka > -26.11,Ca ka > -26.11,Ca ka > -26.11
4,Ca ka > -21.19,Ca ka > -21.19,Ca ka > -21.19,Ca ka > -21.19
...,...,...,...,...
87,background2 <= 0.98,Ar ka + Ag L <= 0.28,Fe kb <= -7.41,background1 > -0.11
88,Ar ka + Ag L > -0.29,Zn kb > -1.05,background2 <= 0.27,Ar ka + Ag L <= 1.07
89,background2 > -0.92,Fe kb <= -6.37,Fe kb <= -6.37,Zn kb > -0.29
90,Class_A,Class_A,Class_A,Class_A


In [10]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_cov_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_cov = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_cov = lrc_summed_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_cov = lrc_summed_df_cov.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
#lrc_summed_unique_df_cov = lrc_summed_unique_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_cov

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Fe ka > -23.71,16.956636,Fe ka,-23.71,>
1,Ca ka > -26.11,8.244088,Ca ka,-26.11,>
2,Fe kb > -7.02,5.715130,Fe kb,-7.02,>
3,Ti ka > -28.53,3.723525,Ti ka,-28.53,>
4,Ca kb > -8.59,1.441617,Ca kb,-8.59,>
5,Cu ka > -10.68,1.169743,Cu ka,-10.68,>
6,Ti kb <= 6.95,0.865477,Ti kb,6.95,<=
7,Cu kb > -1.61,0.252920,Cu kb,-1.61,>
8,Ag ka scattering <= 3.49,0.080663,Ag ka scattering,3.49,<=
9,background3 <= 1.29,0.038303,background3,1.29,<=


In [11]:
# Criar zone_sums_df para dados NÃO pré-processados (Xcalclass)
spectral_zones_original = exp.extract_spectral_zones(Xcalclass, spectral_cuts)
zones_original = exp.aggregate_spectral_zones_pca(spectral_zones_original) # apca aggregation

# Aplicar o mapeamento
lrc_summed_df_cov_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_cov,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_cov_natural

Zona 'Ar ka + Ag L': VE = 9.76%, dim = 29 variáveis
Zona 'Ca ka': VE = 99.20%, dim = 17 variáveis
Zona 'Ca kb': VE = 93.13%, dim = 13 variáveis
Zona 'Ti ka': VE = 98.39%, dim = 19 variáveis
Zona 'Ti kb': VE = 85.13%, dim = 16 variáveis
Zona 'background1': VE = 12.27%, dim = 39 variáveis
Zona 'Fe ka': VE = 99.86%, dim = 25 variáveis
Zona 'Fe kb': VE = 97.91%, dim = 22 variáveis
Zona 'background2': VE = 8.43%, dim = 18 variáveis
Zona 'Cu ka': VE = 98.29%, dim = 20 variáveis
Zona 'Zn ka': VE = 24.14%, dim = 21 variáveis
Zona 'Cu kb': VE = 66.75%, dim = 19 variáveis
Zona 'Zn kb': VE = 7.95%, dim = 30 variáveis
Zona 'background3': VE = 2.98%, dim = 451 variáveis
Zona 'Ag ka scattering': VE = 38.41%, dim = 49 variáveis


,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Fe ka > -23.71,16.956636,Fe ka,-23.71,>,-228.336686,46.0,0.003068,Fe ka > -228.336686
1,Fe ka > -24.15,12.928949,Fe ka,-24.15,>,-232.833909,112.0,0.002150,Fe ka > -232.833909
2,Fe ka > -24.62,10.200160,Fe ka,-24.62,>,-240.227437,43.0,0.003827,Fe ka > -240.227437
3,Ca ka > -26.11,8.244088,Ca ka,-26.11,>,-287.239935,110.0,0.000994,Ca ka > -287.239935
4,Ca ka > -21.19,6.911984,Ca ka,-21.19,>,-238.439137,152.0,0.149595,Ca ka > -238.439137
...,...,...,...,...,...,...,...,...,...
87,background2 <= 0.98,0.004761,background2,0.98,<=,2.540802,242.0,0.002660,background2 <= 2.540802
88,Zn kb > -0.29,0.004610,Zn kb,-0.29,>,-3.715263,178.0,0.000440,Zn kb > -3.715263
89,Fe kb <= -6.37,0.003819,Fe kb,-6.37,<=,-29.329252,103.0,0.002815,Fe kb <= -29.329252
90,Class_A,0.000000,None,None,None,NaN,NaN,NaN,None


## Reconstrução de Thresholds Multivariados

Agora vamos pegar os thresholds dos predicados mais importantes e reconstruí-los como espectros completos.

In [15]:
import plotly.graph_objects as go

n = 3
zone_name = lrc_summed_df_cov_natural.iloc[n]['Zone']
threshold_score = float(lrc_summed_df_cov_natural.iloc[n]['Threshold_Natural'])  # Corrigido: usa n em vez de 7

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
print(f"  - Range de energias: {threshold_spectrum.index[0]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_cov_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Ca ka':
  - Dimensão: 17 variáveis espectrais
  - Range de energias: 3.5 - 3.91
  - Variância explicada pela PC1: 99.20%


In [16]:
# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_pred_conticted_numeric = pd.Series(y_pred_cont) # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred_conticted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=pls_model,
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        #perturbation_value=0,
        perturbation_mode='median', # valores entre 'mean' ou 'min'
        stats_source='full', # full indica usar todas as amostras para calcular estatísticas enquanto que 'fold' usa apenas as amostras do fold atual
        #metric='mean_abs_diff',   # Média com sinal (pode ser negativo)
        aim='regression',
        metric='mean_abs_diff', 
        verbose=True
    )

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graph(
        bags_result=all_results_pert[seed]['bags_result'],
        predicate_ranking_dict=all_results_pert[seed]['pert_results_dict'],
        metric_column='Perturbation',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )

    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
PERTURBATION IMPORTANCE PARA PREDICADOS
Tipo de tarefa (aim): regression
Modo de perturbação: median
Fonte das estatísticas: full
Métrica: mean_abs_diff
Total de folds: 10




,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2,Predicate_pert_Seed_3
0,Ti ka <= 9.34,Ti ka <= -4.69,Ti ka <= -4.69,Ti ka <= -4.69
1,Ti ka <= -4.69,Ti ka <= 9.34,Ti ka > 9.34,Ti ka > 9.34
2,Ti ka > 9.34,Ti ka > 9.34,Ti ka <= 9.34,Ti ka <= 9.34
3,Ti ka <= 19.85,Ti ka <= 19.85,Ti ka <= 19.85,Ti ka <= 19.85
4,Ti ka > -28.53,Ti ka > -28.53,Ti ka > -28.53,Ti ka > -28.53
...,...,...,...,...
87,background2 > -0.92,background2 <= 0.27,background2 > -0.37,background2 > -0.37
88,background2 <= 0.27,background2 > -0.92,background2 > -0.92,background2 > -0.92
89,background2 > 0.27,background2 > 0.27,background2 > 0.27,background2 <= 0.27
90,Class_A,Class_A,Class_A,Class_A


In [17]:
all_results_pert[0]['pert_results_dict']['Bag_1']

,Predicate,Perturbation
0,Ti ka <= -4.69,0.342662
1,Ti ka <= 9.34,0.242418
2,Ti ka > 9.34,0.236619
3,Ti ka <= 19.85,0.214347
4,Ti ka > -28.53,0.204792
...,...,...
85,Fe kb <= -6.37,0.000282
86,background2 <= 0.27,0.000280
87,background2 <= -0.37,0.000275
88,Fe kb <= -7.02,0.000274


In [18]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_pert_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_pert = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_pert = lrc_summed_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_pert = lrc_summed_df_pert.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_pert = lrc_summed_unique_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_pert

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Ti ka <= -4.69,6.615926,Ti ka,-4.69,<=
1,Ca ka > -6.65,2.590539,Ca ka,-6.65,>
2,Cu ka > -8.43,0.866052,Cu ka,-8.43,>
3,Ti kb > 2.45,0.809793,Ti kb,2.45,>
4,Ag ka scattering <= -0.18,0.174365,Ag ka scattering,-0.18,<=
5,Fe ka > -23.71,0.170589,Fe ka,-23.71,>
6,Ca kb > -2.67,0.155115,Ca kb,-2.67,>
7,background3 > -2.55,0.079639,background3,-2.55,>
8,Cu kb > -1.61,0.035982,Cu kb,-1.61,>
9,Fe kb > -7.02,0.033291,Fe kb,-7.02,>


In [19]:
# Aplicar o mapeamento
lrc_summed_df_pert_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_pert,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_pert_natural

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Ti ka <= -4.69,6.615926,Ti ka,-4.69,<=,-49.917691,64.0,0.070519,Ti ka <= -49.917691
1,Ti ka <= 9.34,6.021713,Ti ka,9.34,<=,101.580304,11.0,0.037350,Ti ka <= 101.580304
2,Ti ka > 9.34,5.807942,Ti ka,9.34,>,101.580304,11.0,0.037350,Ti ka > 101.580304
3,Ti ka <= 19.85,5.338879,Ti ka,19.85,<=,207.521929,41.0,0.066299,Ti ka <= 207.521929
4,Ti ka > -28.53,4.429488,Ti ka,-28.53,>,-308.117871,246.0,0.007504,Ti ka > -308.117871
...,...,...,...,...,...,...,...,...,...
87,background2 <= 0.27,0.000208,background2,0.27,<=,0.811175,47.0,0.000881,background2 <= 0.811175
88,background2 > -0.92,0.000181,background2,-0.92,>,-2.538256,171.0,0.001040,background2 > -2.538256
89,background2 > 0.27,0.000156,background2,0.27,>,0.811175,47.0,0.000881,background2 > 0.811175
90,Class_B,0.000000,None,None,None,NaN,NaN,NaN,None


In [20]:
import plotly.graph_objects as go

n = 0
zone_name = lrc_summed_df_pert_natural.iloc[n]['Zone'] # o n indica a linha do dataframe lrc_summed_df_pert_natural que queremos acessar para pegar o nome da zona espectral (coluna 'Zone')
threshold_score = float(lrc_summed_df_pert_natural.iloc[n]['Threshold_Natural']) # o iloc acessa a linha n e a coluna 'Threshold_Natural' para pegar o valor do threshold já mapeado para o espaço natural (não pré-processado)

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
#print(f"  - Range de energias: {threshold_spectrum.index[n]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_pert_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Ti ka':
  - Dimensão: 19 variáveis espectrais
  - Variância explicada pela PC1: 98.39%


In [21]:
# Permutation importance baseado em mudança nas predições do modelo PLS
# Medimos a média da diferença absoluta entre as predições originais e as predições
# obtidas após permutar cada variável. Isso fornece uma métrica contínua de importância.
n_repeats = 10
rng = np.random.RandomState(42)
# Predições base do modelo PLS
baseline_pred = pls_model.predict(Xcalclass_prep)
importance_list = []
X_arr = Xcalclass_prep.copy()
for col in Xcalclass_prep.columns:
    diffs = []
    for _ in range(n_repeats):
        X_perm = X_arr.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)
        perm_pred = pls_model.predict(X_perm)
        diffs.append(np.mean(np.abs(baseline_pred - perm_pred)))
    importance_list.append(np.mean(diffs))

permutation_df = pd.DataFrame({
    'energy': Xcalclass_prep.columns,
    'Permutation_importance': importance_list
})
permutation_df.sort_values(by='Permutation_importance', ascending=False, inplace=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in permutation_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
permutation_df['Zone'] = permutation_df['energy'].map(energy_to_zone_vip)
permutation_unique_df = permutation_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
permutation_unique_df = permutation_unique_df.sort_values(by='Permutation_importance', ascending=False)
permutation_unique_df

,energy,Permutation_importance,Zone
0,4.52,0.054103,Ti ka
1,3.73,0.019036,Ca ka
2,4.95,0.007695,Ti kb
3,8.06,0.007410,Cu ka
4,6.46,0.003626,Fe ka
5,4.03,0.003028,Ca kb
6,21.92,0.002719,Ag ka scattering
7,7.09,0.001075,Fe kb
8,8.93,0.000980,Cu kb
9,15.81,0.000543,background3


In [22]:
import numpy as np

max_len = max(
    len(vip_scores_unique_df['Zone']),
    len(reg_vet_unique_df['Zone']),
    len(shap_unique_df['Zone']),
    len(permutation_unique_df['Zone']),
    len(lrc_summed_unique_df_pert['Zone']),
    len(lrc_summed_unique_df_cov['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'VIP_Score': pad_list(vip_scores_unique_df['Zone'], max_len),
    'Reg_Coefficient' : pad_list(reg_vet_unique_df['Zone'], max_len),
    'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'Permutation' : pad_list(permutation_unique_df['Zone'], max_len),
    'LRC_perturbation' : pad_list(lrc_summed_unique_df_pert['Zone'], max_len),
    'LRC_covariance' : pad_list(lrc_summed_unique_df_cov['Zone'], max_len),
})

features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

,VIP_Score,Reg_Coefficient,Shap,Permutation,LRC_perturbation,LRC_covariance
0,Fe ka,Ti ka,Ti ka,Ti ka,Ti ka,Fe ka
1,Ca ka,Ti kb,Ca ka,Ca ka,Ca ka,Ca ka
2,Ti ka,Ag ka scattering,Cu ka,Ti kb,Cu ka,Fe kb
3,Cu ka,Ca ka,Fe ka,Cu ka,Ti kb,Ti ka
4,Fe kb,Cu ka,Ti kb,Fe ka,Ag ka scattering,Ca kb
5,Ca kb,Ca kb,Ag ka scattering,Ca kb,Fe ka,Cu ka
6,Ti kb,background3,Ca kb,Ag ka scattering,Ca kb,Ti kb
7,Ag ka scattering,Cu kb,background3,Fe kb,background3,Cu kb
8,Cu kb,background1,Ar ka + Ag L,Cu kb,Cu kb,Ag ka scattering
9,background3,Ar ka + Ag L,Fe kb,background3,Fe kb,background3


In [23]:
import rbo
rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        # Remove None values from the lists
        list_1 = [x for x in features_importance[method_1].tolist() if x is not None]
        list_2 = [x for x in features_importance[method_2].tolist() if x is not None]
        # Truncate both lists to the same length (minimum of both)
        min_len = min(len(list_1), len(list_2))
        list_1_trunc = list_1[:min_len]
        list_2_trunc = list_2[:min_len]
        rbo_score = rbo.RankingSimilarity(list_1_trunc, list_2_trunc).rbo(p=0.7, k=10)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_comparison.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_comparison

C:\Users\Usuario\AppData\Local\Temp\ipykernel_21868\3097109587.py:16: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Method_1,Method_2,RBO_Score
10,Shap,LRC_perturbation,0.928489
12,Permutation,LRC_perturbation,0.894933
9,Shap,Permutation,0.880482
4,VIP_Score,LRC_covariance,0.879533
6,Reg_Coefficient,Permutation,0.751735
7,Reg_Coefficient,LRC_perturbation,0.722151
5,Reg_Coefficient,Shap,0.680098
1,VIP_Score,Shap,0.473355
2,VIP_Score,Permutation,0.464176
3,VIP_Score,LRC_perturbation,0.436356


In [24]:
lrc_summed_df_cov_natural.to_csv('lrc_cov_natural.csv', index=False, sep=';')
lrc_summed_df_pert_natural.to_csv('lrc_pert_natural.csv', index=False, sep=';')